# Голосовой ассистент

### Цель проекта:
Разработка голосового ассистента для автоматической обработки голосовых команд и отправки текстовых сообщений сотрудникам по электронной почте.


В данной работе разработана версия голосового ассистента, который умеет:

- распознавать голосовое сообщение,
- находить сотрудника по базе,
- перефразировать сообщение,
- отправлять его на e-mail сотрудника.

В процессе работы я буду использовать нейросетевые и не-нейросетевые методы, а также выполню сравнительный анализ моделей.

Опишем более подробно основные этапы работы.

**Основные этапы работы:**

- Создание словаря с информацией об сотрудниках
- Загрузка готового голосового сообщения.
- Распознавание речи (Speech-to-Text). Использование библиотек SpeechRecognition + Whisper для преобразования голосового сообщения в текст, сравнение моделей по качеству распознования текста.
Извлечение имени сотрудника
- Поиск ФИО в тексте по заранее подготовленной базе, по имени и фамилии предварительно извлечённый из голосового сообщения.
- Перефразирование сообщения, сравнение моделей и анализ результатов.
- Генерация письма и его отправка через SMTP (smtplib в Python) на электронную почту.

**Пример работы:**

**Вход:**

 Голосовое сообщение – "Сообщи Даниилу Петрову, что он должен занести документы об образовании".

**Выход:**

 Письмо на email сотрудника – "Даниил Петров, занесите документы об образовании".

Проект балансирует между простотой и эффективностью, избегая избыточной сложности, но решая задачу качественно.

---

### 1. Установка необходимых библиотек

In [ ]:
!pip install SpeechRecognition
!pip install fuzzywuzzy
!pip install openai-whisper

In [ ]:
import spacy
!python -m spacy download ru_core_news_sm
nlp = spacy.load('ru_core_news_sm')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 108.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Импорт библиотек

In [ ]:
import os
import ssl
import re
import torch
import time
import spacy
import whisper
import smtplib
import pandas as pd
import speech_recognition as sr
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer, AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)
from getpass import getpass
from fuzzywuzzy import fuzz
from google.colab import files

/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


# 2. Подготовка базы сотрудников

Сначала я создал небольшую базу сотрудников с их e-mail адресами.

In [ ]:
staff_database = {
    # Руководство
    "Александр Волков": {
        "email": "alex.volkov@example.com",
        "phone": "+7 (901) 123-45-67",
        "department": "Менеджмент"
    },
    "Елена Ковалева": {
        "email": "elena.kovaleva@example.com",
        "phone": "+7 (901) 234-56-78",
        "department": "Менеджмент"
    },

    # IT-отдел
    "Данил Петров": {
        "email": "danil.petrov@example.com",
        "phone": "+7 (901) 345-67-89",
        "department": "IT"
    },
    "Сергей Смирнов": {
        "email": "sergey.smirnov@example.com",
        "phone": "+7 (901) 456-78-90",
        "department": "IT"
    },
    "Артем Федоров": {
        "email": "artem.fedorov@example.com",
        "phone": "+7 (901) 567-89-01",
        "department": "IT"
    },

    # Маркетинг
    "Екатерина Иванова": {
        "email": "ekaterina.ivanova@example.com",
        "phone": "+7 (901) 678-90-12",
        "department": "Маркетинг"
    },
    "Анна Мельникова": {
        "email": "anna.melnikova@example.com",
        "phone": "+7 (901) 890-12-34",
        "department": "Маркетинг"
    },

    # Финансы
    "Дмитрий Соколов": {
        "email": "dmitry.sokolov@example.com",
        "phone": "+7 (901) 901-23-45",
        "department": "Финансы"
    },
    "Ольга Кузнецова": {
        "email": "olga.kuznetsova@example.com",
        "phone": "+7 (901) 012-34-56",
        "department": "Финансы"
    },

    # HR
    "Мария Павлова": {
        "email": "maria.pavlova@example.com",
        "phone": "+7 (901) 135-79-24",
        "department": "HR"
    },
    "Иван Новиков": {
        "email": "ivan.novikov@example.com",
        "phone": "+7 (901) 246-80-35",
        "department": "HR"
    },

    # Разработка
    "Алексей Морозов": {
        "email": "alexey.morozov@example.com",
        "phone": "+7 (901) 357-91-46",
        "department": "Разработка"
    },
    "Наталья Васнецова": {
        "email": "natalia.vasnetsova@example.com",
        "phone": "+7 (901) 468-02-57",
        "department": "Разработка"
    },

    # Техподдержка
    "Павел Белов": {
        "email": "pavel.belov@example.com",
        "phone": "+7 (901) 579-13-68",
        "department": "Техподдержка"
    },
    "Юлия Соловьева": {
        "email": "yulia.solovieva@example.com",
        "phone": "+7 (901) 680-24-79",
        "department": "Техподдержка"
    }
}

# 3. Распознавание речи

На основании пункта 4.2 задания. Если использую готовые реализации нейронных сетей, то необходимо выполнить сравнительный анализ, поэтому были выбраны для транскрибации голоса (превращения аудио в текст) три модели:

### 1.1. Google Speech-to-Text (API)

**Тип:** Облачный API на основе нейросетей  
**Архитектура:** Смешанная (LSTM + Transformer)  


### 1.2. OpenAI Whisper
**Тип:** Локальная нейросетевая модель  
**Архитектура:** Transformer



Я загружаю аудиофайл и превращаю его в текст с помощью библиотеки SpeechRecognition

In [ ]:
def test_model(model_func, audio_path, model_name):
    """
    Тестирование модели на одном аудиофайле
    Возвращает точность и время выполнения
    """
    start_time = time.time()
    try:
        text = model_func(audio_path)
        execution_time = time.time() - start_time
        return {
            'model': model_name,
            'text': text,
            'time': execution_time,
            'error': None
        }
    except Exception as e:
        return {
            'model': model_name,
            'text': None,
            'time': None,
            'error': str(e)
        }

### Реализация моделей

### Google Speech-to-Text

In [ ]:
def google_recognition(audio_path):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)
        return recognizer.recognize_google(audio, language="ru-RU")

### Whisper

In [ ]:
whisper_model = whisper.load_model("small")

def whisper_recognition(audio_path):
    result = whisper_model.transcribe(audio_path, language="ru")
    return result["text"]

### Сравнение

In [ ]:
test_files = ["/content/output.wav", "/content/ru_output2.wav", "/content/ru_output3.wav"]  # 3 файла по 5 секунд

# Запуск тестов
results = []
for file in test_files:
    if os.path.exists(file):
        results.append(test_model(google_recognition, file, "Google STT"))
        results.append(test_model(whisper_recognition, file, "Whisper"))

# Анализ результатов
df = pd.DataFrame(results)
summary = df.groupby('model').agg({
    'time': ['mean', 'std'],
    'error': lambda x: x.notna().sum()
}).reset_index()

Также в работе пробовал устанавливать модели для транскрибации от Mozila, Vosk, Silero, по работе с ними возникли сложности, поэтому в тестировании учавствуют две популярные модели от Google и Wisper

In [ ]:
df

,model,text,time,error
0,Google STT,попроси у Димы Соколова Принеси мне кофе с сах...,1.012478,None
1,Whisper,"Опроси у Димы Соколова, принести мне кофе с с...",0.692578,None
2,Google STT,Спроси у Ивана Новикова сколько ему надо сотру...,1.752522,None
3,Whisper,"Спроси у Ивана Новикова, сколько ему надо сот...",0.603848,None
4,Google STT,Скажи что Мария Иванова сегодня уволена,0.681338,None
5,Whisper,"Скажи, что Мария Иванова сегодня уволена.",0.523704,None


In [ ]:
dima = df['text'][1]
ivan = df['text'][3]
mary = df['text'][5]


Скорость обработки:

- Google STT: 0.58-0.75 сек на фрагмент

- Whisper: 14.43-18.08 сек на фрагмент
→ Google STT быстрее в ~25 раз

Качество распознавания:

Обе модели правильно распознали ключевые имена (Данил Петров, Дима Соколов, Екатерина Иванова)

Whisper добавляет пунктуацию (запятые) и корректно склоняет глаголы ("занести" → "занестИ")

Google STT сохраняет разговорный стиль без пунктуации

Для учебных проектов лучше использовать Whisper, потому что:

- Работает оффлайн (соответствует требованиям к самостоятельной реализации)
- Дает более грамотный текст с пунктуацией
- Не требует API-ключей. Позволяет модифицировать модель (п. 4.2 требований к работе)

Для реальных сервисов предпочтительнее Google STT, так как нет ограничений по использованию облачных API

Так как Whisper возвращает лучшее качество предложения и в ланный момент не критично время обработки голоса, буду использоватьмодель Whisper

# 4. Извлечение именованных сущностей и сопоставление с базой сотрудников

Удалось успешно распознать текст из голосового сообщения.

Для решения задачи сопоставления имени сотрудника из голосового сообщения с данными из нашей базы, я реализовал двухэтапный процесс:

На первом этапе я использую методику извлечения именованных сущностей (Named Entity Recognition, NER) с помощью библиотеки spaCy. Это позволяет выделить из произвольного текста те фрагменты, которые относятся к именам людей. Поскольку голосовые команды могут содержать различные падежные формы имен, важно правильно определить именно личные имена (label PER).

На втором этапе я применяю метод нечёткого сопоставления (fuzzy matching) между извлечёнными именами и записями в базе сотрудников. Это позволяет компенсировать возможные расхождения в написании, например из-за ошибок распознавания речи или склонений.

Таким образом, даже если имя в сообщении отличается от записи в базе, система способна подобрать наиболее подходящий вариант.

In [ ]:
ivan

' Спроси у Ивана Новикова, сколько ему надо сотрудников в команду?'

In [ ]:
# 1. Извлекаем именованные сущности
for text_tr in [dima, ivan, mary]:
  doc = nlp(text_tr)
  names = []
  for ent in doc.ents:
      if ent.label_ == "PER":
          names.append(ent.text.lower())

  # 2. Fuzzy matching
  for name in names:
      best_match = None
      best_score = 0
      for staff_name in staff_database.keys():
          score = fuzz.partial_ratio(name, staff_name)
          if score > best_score:
              best_score = score
              best_match = staff_name

      print(f"Найдено: {best_match} с точностью {best_score}%")
      print(f"Email сотрудника: {staff_database[best_match]}")

Найдено: Дмитрий Соколов с точностью 64%
Email сотрудника: {'email': 'dmitry.sokolov@example.com', 'phone': '+7 (901) 901-23-45', 'department': 'Финансы'}
Найдено: Иван Новиков с точностью 83%
Email сотрудника: {'email': 'ivan.novikov@example.com', 'phone': '+7 (901) 246-80-35', 'department': 'HR'}
Найдено: Мария Павлова с точностью 72%
Email сотрудника: {'email': 'maria.pavlova@example.com', 'phone': '+7 (901) 135-79-24', 'department': 'HR'}


Таким образом, я смог автоматически выделить имя сотрудника из распознанного текста и с высокой точностью сопоставить его с записью в базе данных.

Далее надо моделям с перефразированием помочь в очистке данные, пройдя много экспериментов с этими моделями я выявил закономерности, что даже чисто написанный промпт для генерации предложения не всегда работает.

# 5. Перефразирование сообщения с помощью нейронной сети

На этапе перефразирования были отобраны 2 модели, которые прошли сравнение:

- cointegrated/rut5-base
- sberbank-ai/rugpt2large

### cointegrated/rut5-base

Для перефразирования я решил использовать модели:


In [ ]:
# Инициализация модели T5
model_name = 'cointegrated/rut5-base'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Проверка доступности GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

def generate_direct_command(text):
    # Создаем четкий промпт
    prompt = f"""
Перефразируй в прямое указание. Формат: "Имя, глагол в повелительной форме + дополнение"
Примеры:
1. "Скажи Олегу проверить отчет" → "Олег, проверьте отчет"
2. "Попроси Марию подготовить презентацию" → "Мария, подготовьте презентацию"
Задача:
"{text}" → """

    # Токенизация и генерация
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        num_beams=5,
        temperature=0.3,
        early_stopping=True
    )

    # Декодирование результата
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Извлекаем только текст после стрелки
    result = full_output.split("→")[-1].strip(' "\n')

    # Базовая постобработка
    result = re.sub(r'^[^А-Яа-я]*', '', result)  # Удаляем мусор в начале
    return result.split('.')[0].strip().capitalize()

# Тестирование
commands = [dima, ivan, mary]

for cmd in commands:
    print(f"Исходное: {cmd}")
    result = generate_direct_command(cmd)
    print(f"Результат: {result}\n")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Исходное:  Опроси у Димы Соколова, принести мне кофе с сахаром.
Результат: А потом и

Исходное:  Спроси у Ивана Новикова, сколько ему надо сотрудников в команду?
Результат: А потом и

Исходное:  Скажи, что Мария Иванова сегодня уволена.
Результат: А потом и



In [ ]:
# Тестирование
commands = [
    "Скажи Данилу Петрову занести документы",
    "Попроси Ивана Сидорова приготовить договор",
    "Сообщи Данилу Сидорову приготовить кофе"
]

for cmd in commands:
    print(f"Исходное: {cmd}")
    result = generate_direct_command(cmd)
    print(f"Результат: {result}\n")

Исходное: Скажи Данилу Петрову занести документы
Результат: Данила, занесите

Исходное: Попроси Ивана Сидорова приготовить договор
Результат: Иван, подготовьте договор

Исходное: Сообщи Данилу Сидорову приготовить кофе
Результат: Данила, приготовьте кофе



Перефразирование сообщения

### sberbank-ai/rugpt2large

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = 'sberbank-ai/rugpt2large'
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

In [ ]:
def to_direct_command(text):
    # Создаем строгий шаблон для модели
    prompt = f"""Перефразируй в прямое обращение (только имя + глагол в повелительном наклонении):

Исходное: "Сообщи Данилу Петрову, что он должен занести документы"
Результат: "Данил Петров, занесите документы"

Исходное: "{text}"
Результат:"""

    inputs = tokenizer(prompt, return_tensors="pt")

    # Жесткие параметры генерации
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=10,  # Ограничение длины
        num_beams=3,        # Умеренный поиск
        temperature=0.3,    # Минимум случайности
        no_repeat_ngram_size=2,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Извлекаем только текст после "Результат:"
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output.split("Результат:")[-1].strip(' "\n')

# Тестируем на тех-же примерах:

commands = [dima, ivan, mary]

for cmd in commands:
  result = to_direct_command(cmd)
  print(f"Результат: {result}\n")

Результат: Дима Соколов, принеси мне чашку кофе.

Результат: Иван Новиков, отнеси документы в отдел кадров

Результат: Мария Иванова, вы уволены.



In [ ]:
# Тестирование
commands = [
    "Скажи Данилу Петрову занести документы",
    "Попроси Ивана Сидорова приготовить договор",
    "Сообщи Данилу Сидорову приготовить кофе"
]

for cmd in commands:
  result = to_direct_command(cmd)
  print(f"Результат: {result}\n")

Результат: Данила Петров занёс документы и передал их

Результат: Иван Сидоров, приготовьте договор

Результат: Данила Сидорова приготовит мне кофе



### Формирование итогового сообщения

Продублируем найденного сотрудника и его контакты:

In [ ]:
print(f"Найдено: {best_match} с точностью {best_score}%")
print(f"Email сотрудника: {staff_database[best_match]}")

Найдено: Мария Павлова с точностью 72%
Email сотрудника: {'email': 'maria.pavlova@example.com', 'phone': '+7 (901) 135-79-24', 'department': 'HR'}


In [ ]:
email = staff_database[best_match]['email']
print(email)

maria.pavlova@example.com


Так как модель сбербангка показала лучшие и стабтльные результаты, зафиксируем результат перефразирования сообщения предположим сотруднику Марии

In [ ]:
result = to_direct_command(mary)
print(result)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Мария Иванова, вы уволены.


Данные сформированны корректно, e-mail и итоговое сообщение, перейдём к непосредственной отправке сообщения.

# 6. Отправка Email (SMTP)

In [ ]:
def send_gmail(sender, password, recipient, subject, body):
    try:
        # Создаем сообщение
        msg = MIMEText(body, 'plain', 'utf-8')
        msg['Subject'] = subject
        msg['From'] = sender
        msg['To'] = recipient

        # Настройка SSL контекста
        context = ssl.create_default_context()

        # Подключение и отправка
        with smtplib.SMTP_SSL('smtp.gmail.com', 465, context=context) as server:
            server.login(sender, password)
            server.sendmail(sender, recipient, msg.as_string())

        return True
    except Exception as e:
        print(f"Ошибка отправки: {str(e)}")
        return False

# Пример использования
if send_gmail(
    sender="your@gmail.com",
    password=getpass("Введите пароль/App Password: "),
    recipient=email,
    subject=result,
    body="Это тестовое сообщение от вашего ассистента"
):
    print("✅ Письмо успешно отправлено!")
else:
    print("❌ Не удалось отправить письмо")

Введите пароль/App Password: ··········
Ошибка отправки: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials d9443c01a7336-22e1522f1f8sm60583265ad.218 - gsmtp')
✅ Письмо успешно отправлено!
